# Statistical Exchange of Space and Time

Computer algorithms tend to solve the same problem over a spectrum of space-time trade-offs. 
It's reasonable to expect the same from statistical estimation

In [1]:
## Dense net code initially authored by Google's search engine GenAI on 20 Oct 2024. 
## I've applied minor modifications for generality, but the code worked great on first draft. 

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

## define control 
class DenseNetControl(nn.Module):
    def __init__(self):
        super(DenseNetControl, self).__init__()
        self.features = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        if x.shape[0] == 0: 
            return torch.tensor([])
        x = x.view(x.size(0), -1)
        x = self.features(x)
        return x
    pass 

## define experimental model 
class DenseNetExperimental(nn.Module):
    def __init__(self):
        super(DenseNetExperimental, self).__init__() 
        self.linear = nn.Linear(128, 128)
        self.features = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            self.linear,
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        if x.shape[0] == 0: 
            return torch.tensor([])
        x = x.view(x.size(0), -1)
        x = self.features(x)
        return x
    pass 

## Load MNIST dataset
train_dataset = datasets.MNIST(root='/tmp/data', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root='/tmp/data', train=False, transform=transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [2]:
## Dense net experiment, needs about 1-2 min of compute 

# Initialize the model, loss function and optimizer 
model_control = DenseNetControl() 
criterion = nn.CrossEntropyLoss() 
optimizer = optim.Adam(model_control.parameters(), lr=0.001) 

model_experimental = DenseNetExperimental() 
criterion = nn.CrossEntropyLoss() 
optimizer = optim.Adam(model_experimental.parameters(), lr=0.001) 

# Train the model 
def fit(model):
    num_epochs = 5 
    for epoch in range(num_epochs): 
        for i, (data, target) in enumerate(train_loader): 
            optimizer.zero_grad() 
            output = model(data) 
            loss = criterion(output, target) 
            loss.backward() 
            optimizer.step() 
            if i % 100 == 0:
                print('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'.format(epoch+1, num_epochs, i+1, len(train_loader), loss.item()))
    # Evaluate the model
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    acc = correct / total
    print('Accuracy of the network on the 10000 test images: {} %'.format(100 * correct / total)) 
    return acc 

In [3]:
acc = fit(model_control)
print(acc)

Epoch [1/5], Step [1/938], Loss: 2.3026
Epoch [1/5], Step [101/938], Loss: 2.3153
Epoch [1/5], Step [201/938], Loss: 2.3155
Epoch [1/5], Step [301/938], Loss: 2.2851
Epoch [1/5], Step [401/938], Loss: 2.3118
Epoch [1/5], Step [501/938], Loss: 2.3148
Epoch [1/5], Step [601/938], Loss: 2.3032
Epoch [1/5], Step [701/938], Loss: 2.3118
Epoch [1/5], Step [801/938], Loss: 2.3055
Epoch [1/5], Step [901/938], Loss: 2.2887
Epoch [2/5], Step [1/938], Loss: 2.3043
Epoch [2/5], Step [101/938], Loss: 2.3078
Epoch [2/5], Step [201/938], Loss: 2.2985
Epoch [2/5], Step [301/938], Loss: 2.3033
Epoch [2/5], Step [401/938], Loss: 2.3182
Epoch [2/5], Step [501/938], Loss: 2.2944
Epoch [2/5], Step [601/938], Loss: 2.3077
Epoch [2/5], Step [701/938], Loss: 2.3076
Epoch [2/5], Step [801/938], Loss: 2.3017
Epoch [2/5], Step [901/938], Loss: 2.3074
Epoch [3/5], Step [1/938], Loss: 2.3027
Epoch [3/5], Step [101/938], Loss: 2.3146
Epoch [3/5], Step [201/938], Loss: 2.3055
Epoch [3/5], Step [301/938], Loss: 2.311

In [4]:
acc = fit(model_experimental)
print(acc)

Epoch [1/5], Step [1/938], Loss: 2.3004
Epoch [1/5], Step [101/938], Loss: 0.4070
Epoch [1/5], Step [201/938], Loss: 0.5792
Epoch [1/5], Step [301/938], Loss: 0.5194
Epoch [1/5], Step [401/938], Loss: 0.1901
Epoch [1/5], Step [501/938], Loss: 0.1819
Epoch [1/5], Step [601/938], Loss: 0.1498
Epoch [1/5], Step [701/938], Loss: 0.2310
Epoch [1/5], Step [801/938], Loss: 0.2111
Epoch [1/5], Step [901/938], Loss: 0.3246
Epoch [2/5], Step [1/938], Loss: 0.0681
Epoch [2/5], Step [101/938], Loss: 0.0554
Epoch [2/5], Step [201/938], Loss: 0.2238
Epoch [2/5], Step [301/938], Loss: 0.0333
Epoch [2/5], Step [401/938], Loss: 0.0711
Epoch [2/5], Step [501/938], Loss: 0.4901
Epoch [2/5], Step [601/938], Loss: 0.0733
Epoch [2/5], Step [701/938], Loss: 0.0864
Epoch [2/5], Step [801/938], Loss: 0.1315
Epoch [2/5], Step [901/938], Loss: 0.0707
Epoch [3/5], Step [1/938], Loss: 0.0278
Epoch [3/5], Step [101/938], Loss: 0.1129
Epoch [3/5], Step [201/938], Loss: 0.1504
Epoch [3/5], Step [301/938], Loss: 0.012